In [1]:
import sqlite3
import pandas as pd 
df = pd.read_excel("week12_transactions.xlsx")
df['date'] = pd.to_datetime(df['date'], unit='D', origin='1899-12-30')
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)
pd.read_sql("SELECT * FROM transactions LIMIT 5", conn)

,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01 00:00:00,8,Gasabo,send,1000,0
1,T00002,2026-11-09 00:00:00,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19 00:00:00,8,Muhanga,send,51000,0
3,T00004,2026-11-01 00:00:00,17,Huye,send,79000,0
4,T00005,2026-11-15 00:00:00,12,Huye,send,109000,0


In [4]:
#PART 1
#1.1.a
pd.read_sql("SELECT txn_id, district, amount_rwf FROM transactions WHERE amount_rwf > 100000 AND txn_type='send' ORDER BY amount_rwf DESC LIMIT 5", conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


In [5]:
#1.1.b
pd.read_sql("SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged = 1",conn)

,flagged_count
0,11


In [6]:
#1.2.a
pd.read_sql("SELECT txn_type, COUNT(*) AS n_txns, SUM(amount_rwf) AS total_rwf, AVG(amount_rwf) AS avg_rwf FROM transactions GROUP BY txn_type ORDER BY total_rwf DESC",conn)

,txn_type,n_txns,total_rwf,avg_rwf
0,send,868,62049000,71485.023041
1,paybill,330,29970000,90818.181818
2,receive,489,27024000,55263.803681
3,airtime,313,833000,2661.341853


In [7]:
#1.2.b
pd.read_sql("SELECT district, COUNT(*) AS n_txns, SUM(amount_rwf) AS total_rwf, AVG(amount_rwf) AS avg_rwf FROM transactions GROUP BY district ORDER BY total_rwf DESC",conn)

,district,n_txns,total_rwf,avg_rwf
0,Muhanga,247,21076000,85327.935223
1,Gasabo,255,20738000,81325.490196
2,Huye,251,18736000,74645.418327
3,Nyarugenge,252,18091000,71789.682540
4,Kicukiro,249,11338000,45534.136546
5,Musanze,253,10490500,41464.426877
6,Rubavu,246,10110500,41099.593496
7,Nyagatare,247,9296000,37635.627530


In [10]:
#1.3.a
pd.read_sql("SELECT hour, COUNT(*) AS n_txns FROM transactions GROUP BY hour ORDER BY n_txns DESC",conn)


,hour,n_txns
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169
5,10,168
6,8,168
7,19,158
8,9,144
9,16,38


In [ ]:
#1.3.b
df.groupby("hour").size().sort_values(ascending=False)

In [11]:
#PART 2.1.a
import json
raw='{"city":"kigali", "temps":[21.5, 24.0, 26.2]}'
data = json.loads(raw)
print(data["city"])

kigali


In [12]:
#2.1.b
print(len(data["temps"]))

3


In [13]:
#2.1.c
print(f"{sum(data['temps'])/len(data['temps']):.2f}")

23.90


In [14]:
#2.2.a
try:
    import urllib.request
    url = "https://api.open-meteo.com/v1/forecast?latitude=-1.9403&longitude=29.8739&hourly=temperature_2m&forecast_days=1"
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read())
        print("Live API used")
except Exception:
    with open("week12_api_sample.json") as f:
        data = json.load(f)
    print("Offline fallback used")
temps = data["hourly"]["temperature_2m"]
times = data["hourly"]["time"]
print(len(temps))

Live API used
24


In [15]:
#2.2.b
print(max(temps), min(temps))

23.9 15.6


In [16]:
#2.2.c
print(times[temps.index(max(temps))])

2026-11-19T11:00


In [17]:
#2.3.a
api_df = pd.DataFrame({"time": times, "temp_c": temps})
print(api_df["temp_c"].describe())

count    24.000000
mean     19.133333
std       2.863665
min      15.600000
25%      16.700000
50%      18.250000
75%      21.750000
max      23.900000
Name: temp_c, dtype: float64


In [18]:
print((api_df["temp_c"] >= 25).sum())

0


In [1]:
# #PART 3.1
import os 
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot"
os.environ["Path"] = os.environ["JAVA_HOME"] + r"\bin;"+ os.environ["PATH"]
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName("momo").getOrCreate())
# df.to_csv("week12_transactions.csv", index=False)
sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)
sdf.count()
sdf.printSchema()
sdf.show(5)


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


In [2]:
#PART 3.2.a
sdf.filter(sdf.amount_rwf>100000).count()

451

In [3]:
#PART 3.2.b
sdf.filter(sdf.flagged == 1).show(5)

+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


In [4]:
#PART 3.3.a
(sdf.groupBy("txn_type").sum("amount_rwf").show())


+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



In [5]:
#PART 3.3.b
district_spark = sdf.groupBy("district").sum("amount_rwf").toPandas()
print(district_spark.sort_values("sum(amount_rwf)", ascending=False))

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


     district  sum(amount_rwf)
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
1  Nyarugenge         18091000
0    Kicukiro         11338000
2     Musanze         10490500
7      Rubavu         10110500
3   Nyagatare          9296000


In [6]:
#PART 3.3.c
# Yes, Spark mathes SQL. send = 62049000, Muhanga = 21076000

In [7]:
#BONUS
sdf.createOrReplaceTempView("transactions")
spark.sql("""SELECT txn_type, COUNT(*) AS n_txns, SUM(amount_rwf) AS total_rwf 
FROM transactions 
GROUP BY txn_type 
ORDER BY total_rwf DESC""").show()

+--------+------+---------+
|txn_type|n_txns|total_rwf|
+--------+------+---------+
|    send|   868| 62049000|
| paybill|   330| 29970000|
| receive|   489| 27024000|
| airtime|   313|   833000|
+--------+------+---------+



In [8]:
#PART 4
#1. SQL
#2. Pandas
#3. Spark
#4. API